In [1]:
from pyspark.sql import SparkSession
print("Import Done")

Import Done


In [2]:
spark = (SparkSession.builder
.appName("RDD Test")
.master("local[*]")
.getOrCreate())
print("Done")

c:\Users\kavin\OneDrive\Desktop\Data_engineer\pyspark\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Done


In [3]:
sc = spark.sparkContext

In [4]:
file = sc.textFile('C:\\Users\\kavin\\OneDrive\\Desktop\\Data_engineer\\pyspark\\Employe.csv',2)
print("File Read Done")

File Read Done


In [5]:
file.collect()

['"mp_id,name,dept,city,salary"',
 '"101,Anu,Sales,Chennai,45000"',
 '"102,Ravi,IT,Bengaluru,72000"',
 '"103,Asha,HR,Chennai,39000"',
 '"104,Kiran,IT,Hyderabad,68000"',
 '"105,Meena,Sales,Mumbai,"',
 '"106,Suresh,Finance,Delhi,abc"',
 '"107,Divya,IT,Chennai,81000"',
 '"108,Arjun,HR,Mumbai,42000"']

In [6]:
file = file.map(lambda l : l.replace('"',''))
file.collect()

['mp_id,name,dept,city,salary',
 '101,Anu,Sales,Chennai,45000',
 '102,Ravi,IT,Bengaluru,72000',
 '103,Asha,HR,Chennai,39000',
 '104,Kiran,IT,Hyderabad,68000',
 '105,Meena,Sales,Mumbai,',
 '106,Suresh,Finance,Delhi,abc',
 '107,Divya,IT,Chennai,81000',
 '108,Arjun,HR,Mumbai,42000']

In [10]:
print(file.getNumPartitions())
row_count = file.count()
print(row_count)
first_3_rows = file.take(3)
print(first_3_rows)

2


9
['mp_id,name,dept,city,salary', '101,Anu,Sales,Chennai,45000', '102,Ravi,IT,Bengaluru,72000']


In [11]:
header = file.first()
header

'mp_id,name,dept,city,salary'

In [12]:
data = file.filter(lambda l : l != header)

In [13]:
data.collect()

['101,Anu,Sales,Chennai,45000',
 '102,Ravi,IT,Bengaluru,72000',
 '103,Asha,HR,Chennai,39000',
 '104,Kiran,IT,Hyderabad,68000',
 '105,Meena,Sales,Mumbai,',
 '106,Suresh,Finance,Delhi,abc',
 '107,Divya,IT,Chennai,81000',
 '108,Arjun,HR,Mumbai,42000']

In [14]:
split_data = data.map(lambda l : l.split(','))
split_data.collect()

[['101', 'Anu', 'Sales', 'Chennai', '45000'],
 ['102', 'Ravi', 'IT', 'Bengaluru', '72000'],
 ['103', 'Asha', 'HR', 'Chennai', '39000'],
 ['104', 'Kiran', 'IT', 'Hyderabad', '68000'],
 ['105', 'Meena', 'Sales', 'Mumbai', ''],
 ['106', 'Suresh', 'Finance', 'Delhi', 'abc'],
 ['107', 'Divya', 'IT', 'Chennai', '81000'],
 ['108', 'Arjun', 'HR', 'Mumbai', '42000']]

In [16]:
bad_records = split_data.filter(lambda t : not(t[4].isdigit()))
good_records = split_data.filter(lambda t : t[4].isdigit())


In [17]:
bad_records.collect()

[['105', 'Meena', 'Sales', 'Mumbai', ''],
 ['106', 'Suresh', 'Finance', 'Delhi', 'abc']]

In [18]:
good_records.collect()

[['101', 'Anu', 'Sales', 'Chennai', '45000'],
 ['102', 'Ravi', 'IT', 'Bengaluru', '72000'],
 ['103', 'Asha', 'HR', 'Chennai', '39000'],
 ['104', 'Kiran', 'IT', 'Hyderabad', '68000'],
 ['107', 'Divya', 'IT', 'Chennai', '81000'],
 ['108', 'Arjun', 'HR', 'Mumbai', '42000']]

In [19]:
typed = good_records.map(lambda t: (int(t[0]),t[1].title(),t[2].upper(),t[3],float(t[4])))

In [20]:
typed.collect()

[(101, 'Anu', 'SALES', 'Chennai', 45000.0),
 (102, 'Ravi', 'IT', 'Bengaluru', 72000.0),
 (103, 'Asha', 'HR', 'Chennai', 39000.0),
 (104, 'Kiran', 'IT', 'Hyderabad', 68000.0),
 (107, 'Divya', 'IT', 'Chennai', 81000.0),
 (108, 'Arjun', 'HR', 'Mumbai', 42000.0)]

In [21]:
final = typed.map(lambda t : t + (t[4] * 0.10,t[4] + t[4] * 0.10))
print(final.collect())

[(101, 'Anu', 'SALES', 'Chennai', 45000.0, 4500.0, 49500.0), (102, 'Ravi', 'IT', 'Bengaluru', 72000.0, 7200.0, 79200.0), (103, 'Asha', 'HR', 'Chennai', 39000.0, 3900.0, 42900.0), (104, 'Kiran', 'IT', 'Hyderabad', 68000.0, 6800.0, 74800.0), (107, 'Divya', 'IT', 'Chennai', 81000.0, 8100.0, 89100.0), (108, 'Arjun', 'HR', 'Mumbai', 42000.0, 4200.0, 46200.0)]


In [22]:
by_dept = typed.map(lambda t : (t[2],t[4])).reduceByKey(lambda a,b: a + b)
by_dept.collect()

[('IT', 221000.0), ('HR', 81000.0), ('SALES', 45000.0)]

In [23]:
from pyspark.sql.types import StructType,StructField,IntegerType,StringType,DoubleType
print("Import Done")

Import Done


In [27]:
schema = StructType([
    StructField("mp_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("dept", StringType(), True),
    StructField("city", StringType(), True),
    StructField("salary", DoubleType(), True)
])

In [33]:
df = spark.createDataFrame(typed,schema)
df.printSchema()

root
 |-- mp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- city: string (nullable = true)
 |-- salary: double (nullable = true)



In [34]:
df.collect()

[Row(mp_id=101, name='Anu', dept='SALES', city='Chennai', salary=45000.0),
 Row(mp_id=102, name='Ravi', dept='IT', city='Bengaluru', salary=72000.0),
 Row(mp_id=103, name='Asha', dept='HR', city='Chennai', salary=39000.0),
 Row(mp_id=104, name='Kiran', dept='IT', city='Hyderabad', salary=68000.0),
 Row(mp_id=107, name='Divya', dept='IT', city='Chennai', salary=81000.0),
 Row(mp_id=108, name='Arjun', dept='HR', city='Mumbai', salary=42000.0)]

In [35]:
df.show()

+-----+-----+-----+---------+-------+
|mp_id| name| dept|     city| salary|
+-----+-----+-----+---------+-------+
|  101|  Anu|SALES|  Chennai|45000.0|
|  102| Ravi|   IT|Bengaluru|72000.0|
|  103| Asha|   HR|  Chennai|39000.0|
|  104|Kiran|   IT|Hyderabad|68000.0|
|  107|Divya|   IT|  Chennai|81000.0|
|  108|Arjun|   HR|   Mumbai|42000.0|
+-----+-----+-----+---------+-------+

